# Feature Engineering Experiments (No Leakage)

This notebook compares baseline features against a minimal engineered set to improve rent prediction accuracy while avoiding target leakage.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.svm import SVR
from sklearn.linear_model import Ridge
from sklearn.compose import TransformedTargetRegressor
from sklearn.model_selection import KFold, cross_validate

pd.set_option("display.max_columns", 200)

project_root = Path.cwd().resolve()
while project_root != project_root.parent and not (project_root / "data_cleaning").exists():
    project_root = project_root.parent

if not (project_root / "data_cleaning").exists():
    raise FileNotFoundError("Could not locate project root containing data_cleaning/.")

candidate_paths = [
    project_root / "data_cleaning" / "Khmer24_cleaned_v4.csv",
    project_root / "data_cleaning" / "Khmer24_cleaned_auto.csv",
    project_root / "data" / "Khmer24_features.csv",
]

input_path = next((p for p in candidate_paths if p.exists()), None)
if input_path is None:
    raise FileNotFoundError(f"No input file found. Checked: {[str(p) for p in candidate_paths]}")

print(f"Using input file: {input_path}")

Using input file: C:\Users\USer\Documents\Year3\Data_Science_Project\DS_website\Property-Price-Prediction\data_cleaning\Khmer24_cleaned_auto.csv


In [ ]:
def read_csv_robust(path_obj: Path) -> pd.DataFrame:
    last_err = None
    for enc in ("utf-8", "utf-8-sig", "cp1252", "latin-1"):
        try:
            return pd.read_csv(path_obj, encoding=enc)
        except UnicodeDecodeError as err:
            last_err = err
    raise last_err

df = read_csv_robust(input_path).copy()

for col in ["rent_price_usd", "size_sqm", "bedrooms", "bathrooms"]:
    if col not in df.columns:
        df[col] = np.nan

df["rent_price_usd"] = (
    df["rent_price_usd"]
    .astype(str)
    .str.replace(r"[\$,]|/month|permonth", "", regex=True)
    .str.strip()
)
df["rent_price_usd"] = pd.to_numeric(df["rent_price_usd"], errors="coerce")

for col in ["size_sqm", "bedrooms", "bathrooms"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df = df.dropna(subset=["rent_price_usd"]).copy()
df = df[df["rent_price_usd"] > 0].copy()

for col in ["size_sqm", "bedrooms", "bathrooms"]:
    df[col] = df[col].fillna(df[col].median())

df["size_sqm"] = df["size_sqm"].clip(lower=8, upper=2000)
df["bedrooms"] = df["bedrooms"].clip(lower=0, upper=20)
df["bathrooms"] = df["bathrooms"].clip(lower=1, upper=20)

q01 = df["rent_price_usd"].quantile(0.01)
q99 = df["rent_price_usd"].quantile(0.99)
df = df[(df["rent_price_usd"] >= q01) & (df["rent_price_usd"] <= q99)].copy()

print(f"Rows used: {len(df):,}")
print(df[["rent_price_usd", "size_sqm", "bedrooms", "bathrooms"]].describe().round(2))

Rows used: 1,085
       rent_price_usd  size_sqm  bedrooms  bathrooms
count         1085.00   1085.00   1085.00    1085.00
mean          3305.77    154.44      3.64       3.29
std          16687.47     64.65      1.40       0.98
min            200.00     25.00      1.00       1.00
25%            600.00    120.00      3.00       3.00
50%           1100.00    130.00      4.00       3.00
75%           2500.00    220.00      4.00       4.00
max         239000.00    280.00     20.00       5.00


In [3]:
# Baseline features (same idea as original simple model input)
X_base = df[["size_sqm", "bedrooms", "bathrooms"]].copy()

# Minimal engineered feature set (no target leakage)
fe = df.copy()
fe["log_size_sqm"] = np.log1p(fe["size_sqm"])
fe["bath_per_bed"] = np.where(fe["bedrooms"] > 0, fe["bathrooms"] / fe["bedrooms"], fe["bathrooms"])
fe["total_rooms"] = fe["bedrooms"] + fe["bathrooms"]
fe["size_per_room"] = fe["size_sqm"] / np.clip(fe["total_rooms"], 1, None)

district = fe.get("district", pd.Series("unknown", index=fe.index)).astype(str).str.lower().str.strip()
fe["district_freq"] = district.map(district.value_counts(normalize=True)).fillna(0.5)

ptype = fe.get("property_type", pd.Series("Unclassified", index=fe.index)).astype(str).str.title().str.strip()
type_dummies = pd.get_dummies(ptype, prefix="type")

X_fe = pd.concat([
    fe[["size_sqm", "bedrooms", "bathrooms", "log_size_sqm", "bath_per_bed", "total_rooms", "size_per_room", "district_freq"]],
    type_dummies
], axis=1).astype(float)

y = fe["rent_price_usd"].astype(float)

print(f"Baseline features: {X_base.shape[1]}")
print(f"Engineered features: {X_fe.shape[1]}")

Baseline features: 3
Engineered features: 21


In [ ]:
def evaluate_setup(X: pd.DataFrame, y: pd.Series, setup_name: str) -> pd.DataFrame:
    cv = KFold(n_splits=5, shuffle=True, random_state=42)

    models = {
        "Ridge": Ridge(alpha=3.0),
        "RandomForest": RandomForestRegressor(
            n_estimators=500, max_depth=14, min_samples_leaf=2, random_state=42, n_jobs=-1
        ),
        "HistGradientBoosting": HistGradientBoostingRegressor(
            max_iter=500, learning_rate=0.05, max_depth=8, random_state=42
        ),
        "SVR": SVR(kernel="rbf", C=80, epsilon=0.06, gamma="scale"),
    }

    rows = []
    for name, base_model in models.items():
        model = TransformedTargetRegressor(
            regressor=base_model,
            func=np.log1p,
            inverse_func=np.expm1,
            check_inverse=False,
        )

        scores = cross_validate(
            model,
            X,
            y,
            cv=cv,
            scoring={
                "r2": "r2",
                "rmse": "neg_root_mean_squared_error",
                "mae": "neg_mean_absolute_error",
            },
            n_jobs=-1,
            return_train_score=False,
        )

        rows.append({
            "setup": setup_name,
            "model": name,
            "cv_r2_mean": float(np.mean(scores["test_r2"])),
            "cv_rmse_mean": float(-np.mean(scores["test_rmse"])),
            "cv_mae_mean": float(-np.mean(scores["test_mae"])),
        })

    return pd.DataFrame(rows)

result_base = evaluate_setup(X_base, y, "baseline")
result_fe = evaluate_setup(X_fe, y, "engineered")
result = pd.concat([result_base, result_fe], ignore_index=True)
result = result.sort_values(["cv_r2_mean", "cv_rmse_mean"], ascending=[False, True]).reset_index(drop=True)

print("Model ranking (higher R2 is better, lower RMSE/MAE is better):")
result.round(4)

Model ranking (higher R2 is better, lower RMSE/MAE is better):


,setup,model,cv_r2_mean,cv_rmse_mean,cv_mae_mean
0,engineered,RandomForest,0.0493,15886.7168,2410.0802
1,engineered,HistGradientBoosting,0.0152,16223.2871,2424.2647
2,baseline,RandomForest,-0.0137,16493.7874,2442.2711
3,engineered,Ridge,-0.0138,16492.9275,2470.6284
4,baseline,HistGradientBoosting,-0.0138,16494.3371,2441.9363
5,baseline,SVR,-0.0154,16506.1746,2455.2915
6,engineered,SVR,-0.0157,16507.4046,2451.6416
7,baseline,Ridge,-0.0175,16517.9687,2519.4323


## How To Promote Winning Features

1. Keep only feature sets that improve both CV R2 and CV RMSE against baseline.
2. Add those features to ml/preprocess.py and backend/app/route/predict.py in the same order.
3. Retrain with ml/train_model.py and re-check ml/evaluate_models.py.